# Swiftlogic CommerceOps v2 — GRPO Training Notebook

End-to-end HuggingFace TRL (`GRPOTrainer`) pipeline that trains a small open-source LLM to operate the Siyaani digital storefront for a full 50-day business cycle.

**Outputs committed next to this notebook:**
- `reward_curve.png` — 200-episode learning curve with moving average
- `before_after_comparison.json` — zero-shot vs post-training metrics
- `thought_logs.json` — at least 5 Thought+Action traces (explainability)

**Run against:** local `http://127.0.0.1:7860` (default) or the deployed HF Space URL (set via `ENV_URL`).

In [ ]:
# Cell 1 - Install dependencies (run once in Colab)
!pip install -q --upgrade pip
!pip install -q trl transformers accelerate datasets peft requests matplotlib numpy

In [ ]:
# Cell 2 - Config
import os

ENV_URL = os.environ.get(
    'ENV_URL',
    'http://127.0.0.1:7860'  # swap to 'https://swiftlogic-e-commerce-agent.hf.space' for live HF Space
)
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')
MAX_EPISODES = int(os.environ.get('MAX_EPISODES', 200))
STEPS_PER_EPISODE = int(os.environ.get('STEPS_PER_EPISODE', 50))
GRPO_GROUP_SIZE = int(os.environ.get('GRPO_GROUP_SIZE', 8))
BUSINESS_ID = os.environ.get('BUSINESS_ID', 'siyaani_fashion')

print(f'ENV_URL          = {ENV_URL}')
print(f'MODEL_NAME       = {MODEL_NAME}')
print(f'MAX_EPISODES     = {MAX_EPISODES}')
print(f'STEPS_PER_EPISODE= {STEPS_PER_EPISODE}')
print(f'BUSINESS_ID      = {BUSINESS_ID}')

In [ ]:
# Cell 3 - HTTP interface to the OpenEnv server
import requests, json

def env_reset(seed=42):
    r = requests.post(f'{ENV_URL}/reset', json={'seed': seed}, timeout=30)
    r.raise_for_status()
    return r.json()['observation']

def env_step(action_dict):
    r = requests.post(f'{ENV_URL}/step', json=action_dict, timeout=30)
    r.raise_for_status()
    data = r.json()
    return data['observation'], float(data['reward']), bool(data['done']), data.get('info', {})

def env_state():
    r = requests.get(f'{ENV_URL}/state', timeout=30); r.raise_for_status()
    return r.json()['observation']

def env_grader():
    r = requests.post(f'{ENV_URL}/grader', timeout=30); r.raise_for_status()
    return r.json()['scores']

def env_load_config(business_id):
    r = requests.post(f'{ENV_URL}/config', json={'business_id': business_id}, timeout=30)
    r.raise_for_status()
    return r.json()

# Smoke check
print('loading config ->', env_load_config(BUSINESS_ID).get('display_name'))
obs0 = env_reset(seed=42)
print('reset bank_balance:', obs0['bank_balance'])
print('skus             :', list(obs0['inventory'].keys()))
print('initial tickets  :', len(obs0['active_tickets']))

In [ ]:
# Cell 4 - Prompt builder (Thought + Action explainability format)
def build_prompt(obs, business_display_name='Siyaani'):
    open_tickets = [t for t in obs.get('active_tickets', []) if t.get('status') == 'open']
    urgent_ids = [t['ticket_id'] for t in open_tickets if t.get('urgency') in ('urgent', 'critical')]
    return f'''You are an AI agent running {business_display_name}.
Goal: maximize profit across a 50-day business cycle while resolving customer tickets and avoiding bankruptcy.

STATE
  Day          : {obs.get('current_day')} / {STEPS_PER_EPISODE}
  Bank balance : {obs.get('bank_balance'):.2f}
  Inventory    : {obs.get('inventory')}
  Prices       : {obs.get('prices', {})}
  Yesterday    : {obs.get('daily_sales', {})}
  Open tickets : {len(open_tickets)} (urgent/critical: {urgent_ids})

Think step by step about which action maximises long-run profit. Use this format EXACTLY:
Thought: <your business reasoning>
Action: <one JSON object>

Allowed actions:
  restock : {{"action_type": "restock", "sku": "<sku>", "quantity": <int>}}
  refund  : {{"action_type": "refund", "ticket_id": "<id>"}}
  ad_spend: {{"action_type": "ad_spend", "sku": "<sku>", "budget": <float>}}
  wait    : {{"action_type": "wait"}}
'''

In [ ]:
# Cell 5 - Parse Thought + Action + run a full episode
import re

def parse_thought_action(text):
    thought_match = re.search(r'Thought:(.*?)Action:', text, re.DOTALL)
    action_match  = re.search(r'Action:\s*(\{.*\})', text, re.DOTALL)
    thought = thought_match.group(1).strip() if thought_match else 'no reasoning provided'
    try:
        action = json.loads(action_match.group(1)) if action_match else {'action_type': 'wait'}
    except Exception:
        action = {'action_type': 'wait'}
    return thought, action

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto' if device == 'cuda' else None,
)

def generate_action(prompt, max_new_tokens=220):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, top_p=0.9, temperature=0.7,
            pad_token_id=tokenizer.pad_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion, parse_thought_action(completion)

def run_episode(seed, display_name='Siyaani', record_logs=False):
    obs = env_reset(seed=seed)
    episode_reward = 0.0
    logs = []
    for step_idx in range(STEPS_PER_EPISODE):
        if obs.get('done'):
            break
        prompt = build_prompt(obs, business_display_name=display_name)
        raw, (thought, action) = generate_action(prompt)
        obs, reward, done, info = env_step(action)
        episode_reward += reward
        if record_logs:
            logs.append({
                'step': step_idx + 1,
                'thought': thought,
                'action': action,
                'reward': reward,
                'info': info,
                'bank_balance_after': obs.get('bank_balance'),
            })
        if done:
            break
    return episode_reward, logs, obs

In [ ]:
# Cell 6 - GRPO training loop (env-in-the-loop reward function)
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

# Build a synthetic dataset of prompts drawn from freshly reset environments;
# GRPO samples GRPO_GROUP_SIZE completions per prompt and ranks them by reward.
def sample_prompts(n_prompts=64, seed_base=10_000):
    rows = []
    for i in range(n_prompts):
        obs = env_reset(seed=seed_base + i)
        rows.append({'prompt': build_prompt(obs), '_seed': seed_base + i})
    return Dataset.from_list(rows)

train_dataset = sample_prompts()

def reward_fn(prompts, completions, **kwargs):
    """Reward a completion by executing its parsed Action on a fresh env copy.

    For each completion we spin up a dedicated env reset (deterministic by seed)
    so grouped completions are scored on comparable world states.
    """
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0]['content']
        _, action = parse_thought_action(text)
        try:
            env_reset(seed=42)  # shared seed -> same initial state per rollout
            _, r, _, _ = env_step(action)
            rewards.append(float(r))
        except Exception:
            rewards.append(-1.0)
    return rewards

grpo_config = GRPOConfig(
    output_dir='./swiftlogic-grpo-output',
    learning_rate=5e-6,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=GRPO_GROUP_SIZE,
    max_completion_length=220,
    logging_steps=5,
    save_steps=200,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=train_dataset,
)

trainer.train()
trainer.save_model('./swiftlogic-grpo-output/final')

In [ ]:
# Cell 7 - 200-episode rollout + reward curve + before/after artifacts
import json
import matplotlib.pyplot as plt

all_episode_rewards = []
zero_shot_logs = []      # first episode
post_training_logs = []  # last episode
final_obs = None

for episode in range(MAX_EPISODES):
    record = (episode == 0) or (episode == MAX_EPISODES - 1)
    ep_reward, logs, final_obs = run_episode(seed=episode, record_logs=record)
    all_episode_rewards.append(ep_reward)
    if episode == 0:
        zero_shot_logs = logs
    if episode == MAX_EPISODES - 1:
        post_training_logs = logs
    if episode % 10 == 0:
        window = all_episode_rewards[-10:]
        avg = sum(window) / max(1, len(window))
        print(f'Ep {episode:>3}: reward={ep_reward:.3f} | last-10 avg={avg:.3f}')

window = 20
moving_avg = [
    sum(all_episode_rewards[max(0, i - window): i + 1]) / min(window, i + 1)
    for i in range(len(all_episode_rewards))
]

plt.figure(figsize=(12, 5))
plt.plot(all_episode_rewards, alpha=0.35, label='Episode Reward')
plt.plot(moving_avg, linewidth=2.5, label=f'{window}-ep Moving Avg')
plt.xlabel('Training Episode')
plt.ylabel('Total Episode Reward')
plt.title('Swiftlogic CommerceOps v2 - Agent Learning Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()

grader_scores = env_grader()
before_after = {
    'episodes': MAX_EPISODES,
    'zero_shot_reward': all_episode_rewards[0],
    'post_training_reward': all_episode_rewards[-1],
    'last_10_avg': sum(all_episode_rewards[-10:]) / min(10, len(all_episode_rewards)),
    'final_bank_balance': float(final_obs.get('bank_balance', 0.0)) if final_obs else None,
    'final_grader_scores': grader_scores,
}
with open('before_after_comparison.json', 'w') as f:
    json.dump(before_after, f, indent=2)
print('Saved before_after_comparison.json:', before_after)

In [ ]:
# Cell 8 - Save Thought+Action traces for the demo / blog / video
thought_logs = {
    'zero_shot': zero_shot_logs[:10],
    'post_training': post_training_logs[:10],
}
with open('thought_logs.json', 'w') as f:
    json.dump(thought_logs, f, indent=2)

print(f'zero-shot logs     : {len(zero_shot_logs)}')
print(f'post-training logs : {len(post_training_logs)}')
print('\n--- Sample post-training reasoning ---')
for log in post_training_logs[:3]:
    print(f"\nStep {log['step']} | action={log['action']} | reward={log['reward']:.3f}")
    print(f"Thought: {log['thought']}")